# MergeSortBottomUp (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/MergeSortBottomUp_cpp.ipynb)

Chapter 18 — Recursive versus iterative algorithms. Runnable notebook of `MergeSortBottomUp`: the algorithm in C++, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/18-recursive-vs-iterative/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/MergeSortBottomUp_python.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/MergeSortBottomUp_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative_cpp.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Each C++ program is written to a file by a `%%writefile` cell, then compiled with `g++` and run in the next cell: everything happens inside Colab.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `FibRec` → `FibRec` (PascalCase).

## Recursive MergeSort and MergeSortBottomUp

**MergeSortBottomUp**$(a,n)$ — *Input*: an array $a[1..n]$ of natural numbers. *Output*: the array sorted in non-decreasing order. The same splitting as Merge Sort, but **without recursion**: first we merge pairs of sub-arrays of length $1$, then of length $2$, $4$, … with the procedure `Merge` of chapter 17 (repeated here). It executes $\lceil\log_2 n\rceil$ passes and $\Theta(n\log_2 n)$ steps, like the recursive version, and does not use the call stack.

First the example of the chapter $a=[5,2,4,6,1,3]$, pass by pass (indices are from 0). Then the comparison, on pseudo-random arrays (same generator in all languages, seed $12345$, values between $1$ and $10^6$), of the number of **comparisons** between elements of `InsertionSort`, `MergeSort` and `MergeSortBottomUp`: $\Theta(n^2)$ against $\Theta(n\log_2 n)$, with the two versions of Merge Sort practically equal ("OK" means that the three algorithms give the same sorted array), and the measured times.

In [1]:
%%writefile bottom_up.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long comparisons = 0;   // comparisons between elements

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// Merge(a, l, m, r) of chapter 17
void Merge(int *a, int l, int m, int r)
{
    int n1 = m - l + 1;
    int n2 = r - m;
    vector<int> L(n1), R(n2);
    for (int i = 0; i < n1; i++)
    {
        L[i] = a[l + i];
    }
    for (int j = 0; j < n2; j++)
    {
        R[j] = a[m + 1 + j];
    }
    int i = 0, j = 0, k = l;
    while (i < n1 && j < n2)
    {
        comparisons++;
        if (L[i] <= R[j])
        {
            a[k] = L[i];
            i++;
        }
        else
        {
            a[k] = R[j];
            j++;
        }
        k++;
    }
    while (i < n1)
    {
        a[k] = L[i];
        i++;
        k++;
    }
    while (j < n2)
    {
        a[k] = R[j];
        j++;
        k++;
    }
}

void MergeSort(int *a, int l, int r)
{
    if (l < r)
    {
        int m = l + (r - l) / 2;
        MergeSort(a, l, m);
        MergeSort(a, m + 1, r);
        Merge(a, l, m, r);
    }
}

// MergeSortBottomUp(a, n): sorts a[0..n-1] without recursion
void MergeSortBottomUp(int *a, int n)
{
    int width = 1;
    while (width < n)
    {
        int l = 0;
        while (l + width < n)                   // pseudo-code: l + width <= n
        {
            int m = l + width - 1;
            int r = min(l + 2 * width - 1, n - 1);
            Merge(a, l, m, r);
            l = l + 2 * width;
        }
        width = 2 * width;
    }
}

void MergeSortBottomUpTrace(int *a, int n)
{
    int width = 1;
    int passes = 0;
    while (width < n)
    {
        int l = 0;
        string parts;
        while (l + width < n)
        {
            int m = l + width - 1;
            int r = min(l + 2 * width - 1, n - 1);
            Merge(a, l, m, r);
            if (!parts.empty()) parts += ", ";
            parts += "a[" + to_string(l) + ".." + to_string(m) + "] with a[" + to_string(m + 1) + ".." + to_string(r) + "]";
            l = l + 2 * width;
        }
        passes++;
        printf("width=%d: merge %s => %s\n", width, parts.c_str(), Fmt(a, n).c_str());
        width = 2 * width;
    }
    printf("passes = %d\n", passes);
}

// InsertionSort(a, n) (chapter 3), with the count of comparisons
void InsertionSort(int *a, int n)
{
    for (int j = 1; j < n; j++)
    {
        int key = a[j];
        int i = j - 1;
        while (i >= 0)
        {
            comparisons++;
            if (a[i] > key)
            {
                a[i + 1] = a[i];
                i = i - 1;
            }
            else
            {
                break;
            }
        }
        a[i + 1] = key;
    }
}

// Linear congruential generator: same numbers in Python, C++ and Java
vector<int> RandomArray(int n, unsigned long long seed = 12345)
{
    unsigned long long x = seed;
    vector<int> a(n);
    for (int i = 0; i < n; i++)
    {
        x = (x * 1103515245ULL + 12345ULL) % 2147483648ULL;
        a[i] = 1 + (int)(x % 1000000ULL);
    }
    return a;
}

double NowMs()
{
    return chrono::duration<double, milli>(chrono::steady_clock::now().time_since_epoch()).count();
}

#ifndef NOMAIN
int main()
{
    int a[] = {5, 2, 4, 6, 1, 3};
    printf("a = %s   n = %d\n", Fmt(a, 6).c_str(), 6);
    MergeSortBottomUpTrace(a, 6);

    printf("\n");
    printf("%6s %14s %10s %9s  comparisons\n", "n", "InsertionSort", "MergeSort", "BottomUp");
    vector<array<double, 4>> times;
    for (int n : {1000, 2000, 4000})
    {
        vector<int> base = RandomArray(n);
        vector<int> x1 = base, x2 = base, x3 = base;
        comparisons = 0;
        double t0 = NowMs(); InsertionSort(x1.data(), n); double t1 = NowMs() - t0;
        long long c1 = comparisons;
        comparisons = 0;
        t0 = NowMs(); MergeSort(x2.data(), 0, n - 1); double t2 = NowMs() - t0;
        long long c2 = comparisons;
        comparisons = 0;
        t0 = NowMs(); MergeSortBottomUp(x3.data(), n); double t3 = NowMs() - t0;
        long long c3 = comparisons;
        vector<int> ord = base;
        sort(ord.begin(), ord.end());
        const char *ok = (x1 == x2 && x2 == x3 && x3 == ord) ? "OK" : "ERROR";
        printf("%6d %14lld %10lld %9lld  %s\n", n, c1, c2, c3, ok);
        times.push_back({(double)n, t1, t2, t3});
    }
    printf("\n");
    printf("Measured times (they depend on the machine):\n");
    for (auto &t : times)
    {
        printf("n=%-6d InsertionSort %10.3f ms   MergeSort %8.3f ms   MergeSortBottomUp %8.3f ms\n", (int)t[0], t[1], t[2], t[3]);
    }
    return 0;
}
#endif

Writing bottom_up.cpp


In [2]:
!g++ -std=c++17 -O2 -o bottom_up bottom_up.cpp && ./bottom_up

a = [5, 2, 4, 6, 1, 3]   n = 6
width=1: merge a[0..0] with a[1..1], a[2..2] with a[3..3], a[4..4] with a[5..5] => [2, 5, 4, 6, 1, 3]
width=2: merge a[0..1] with a[2..3] => [2, 4, 5, 6, 1, 3]
width=4: merge a[0..3] with a[4..5] => [1, 2, 3, 4, 5, 6]
passes = 3

     n  InsertionSort  MergeSort  BottomUp  comparisons
  1000         248782       8716      8740  OK
  2000        1010300      19415     19438  OK
  4000        4027558      42874     42876  OK

Measured times (they depend on the machine):
n=1000   InsertionSort      0.241 ms   MergeSort    0.131 ms   MergeSortBottomUp    0.121 ms
n=2000   InsertionSort      0.962 ms   MergeSort    0.248 ms   MergeSortBottomUp    0.227 ms
n=4000   InsertionSort      3.715 ms   MergeSort    0.553 ms   MergeSortBottomUp    0.474 ms


**Try it yourself.** Change $n$ (with $n=8000$ the comparisons of `InsertionSort` quadruple with respect to $n=4000$, those of Merge Sort little more than double).

In [3]:
%%writefile prova_bottom_up.cpp
#define NOMAIN
#include "bottom_up.cpp"

int main()
{
    // Try n = 8000: the comparisons of InsertionSort quadruple with respect to n = 4000, those of Merge Sort little more than double.
    int n = 8000;
    vector<int> base = RandomArray(n);
    vector<int> x1 = base, x2 = base;
    comparisons = 0;
    InsertionSort(x1.data(), n);
    long long c1 = comparisons;
    comparisons = 0;
    MergeSortBottomUp(x2.data(), n);
    printf("%lld %lld %s\n", c1, comparisons, x1 == x2 ? "True" : "False");
    return 0;
}

Writing prova_bottom_up.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_bottom_up prova_bottom_up.cpp && ./prova_bottom_up

16115652 93756 True


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).